# 4.2. Верификация QUBO-представления

# 4.2.1. Проверка построения направленного рыночного графа

На первом этапе проверяется корректность построения матрицы весов направленного рыночного графа.

Для пары активов $(i,j)$ ребро $i \to j$ интерпретируется как:

- short по активу $i$;
- long по активу $j$.

Вес ребра задаётся формулой

$$
w_{ij} = S_{ij}(\mathrm{ask}_j - \mathrm{bid}_i),
$$

где $S_{ij}$ - мера похожести активов, $\mathrm{ask}_j$ - цена покупки актива $j$, а $\mathrm{bid}_i$ - цена продажи актива $i$.

Если $w_{ij}<0$, то покупка актива $j$ относительно дешевле продажи актива $i$ с учётом похожести активов. Поэтому отрицательные веса интерпретируются как потенциально интересные торговые направления.

In [30]:
import numpy as np
import pandas as pd
import sys

sys.path.append('курсовая_2')
from market_graph import build_market_weights

In [31]:
symbols = ["A", "B", "C"]

ask = np.array([1.01, 0.98, 1.05])
bid = np.array([1.00, 0.97, 1.04])

sim = np.array([
    [0.0, 0.8, 0.2],
    [0.8, 0.0, 0.5],
    [0.2, 0.5, 0.0],
])

pd.DataFrame({
    "symbol": symbols,
    "bid": bid,
    "ask": ask,
})

,symbol,bid,ask
0,A,1.00,1.01
1,B,0.97,0.98
2,C,1.04,1.05


In [32]:
w = build_market_weights(ask=ask, bid=bid, sim=sim)

w_df = pd.DataFrame(
    w,
    index=["dummy"] + symbols,
    columns=["dummy"] + symbols,
)

w_df

,dummy,A,B,C
dummy,0.0,0.000,0.000,0.00
A,0.0,0.000,-0.016,0.01
B,0.0,0.032,0.000,0.04
C,0.0,-0.006,-0.030,0.00


In [33]:
manual_w_A_B = sim[0, 1] * (ask[1] - bid[0])
code_w_A_B = w[1, 2]  # node 1 = A, node 2 = B

manual_w_A_B, code_w_A_B

(np.float64(-0.016000000000000014), np.float64(-0.016000000000000014))

In [34]:
assert np.isclose(manual_w_A_B, code_w_A_B)

In [35]:
manual_w_B_A = sim[1, 0] * (ask[0] - bid[1])
code_w_B_A = w[2, 1]

manual_w_B_A, code_w_B_A

(np.float64(0.03200000000000003), np.float64(0.03200000000000003))

In [36]:
assert np.isclose(manual_w_B_A, code_w_B_A)
assert not np.isclose(w[1, 2], w[2, 1])

print("w[A,B] =", w[1, 2])
print("w[B,A] =", w[2, 1])

w[A,B] = -0.016000000000000014
w[B,A] = 0.03200000000000003


In [37]:
dummy_out = w[0, :]
dummy_in = w[:, 0]

dummy_out, dummy_in

(array([0., 0., 0., 0.]), array([0., 0., 0., 0.]))

In [38]:
assert np.allclose(dummy_out, 0.0)
assert np.allclose(dummy_in, 0.0)

print("Dummy node check passed.")

Dummy node check passed.


In [39]:
diag = np.diag(w)

diag

array([0., 0., 0., 0.])

In [40]:
assert np.allclose(diag, 0.0)

print("Diagonal check passed.")

Diagonal check passed.


In [41]:
edge_rows = []

for i in range(1, len(symbols) + 1):
    for j in range(1, len(symbols) + 1):
        if i == j:
            continue

        short_symbol = symbols[i - 1]
        long_symbol = symbols[j - 1]

        edge_rows.append({
            "edge": f"{short_symbol} -> {long_symbol}",
            "short": short_symbol,
            "long": long_symbol,
            "similarity": sim[i - 1, j - 1],
            "bid_short": bid[i - 1],
            "ask_long": ask[j - 1],
            "ask_long_minus_bid_short": ask[j - 1] - bid[i - 1],
            "weight": w[i, j],
            "is_negative": w[i, j] < 0,
        })

edges_df = pd.DataFrame(edge_rows)

edges_df.sort_values("weight")

,edge,short,long,similarity,bid_short,ask_long,ask_long_minus_bid_short,weight,is_negative
5,C -> B,C,B,0.5,1.04,0.98,-0.06,-0.030,True
0,A -> B,A,B,0.8,1.00,0.98,-0.02,-0.016,True
4,C -> A,C,A,0.2,1.04,1.01,-0.03,-0.006,True
1,A -> C,A,C,0.2,1.00,1.05,0.05,0.010,False
2,B -> A,B,A,0.8,0.97,1.01,0.04,0.032,False
3,B -> C,B,C,0.5,0.97,1.05,0.08,0.040,False


In [42]:
negative_edges_df = edges_df[edges_df["is_negative"]].sort_values("weight")

negative_edges_df

,edge,short,long,similarity,bid_short,ask_long,ask_long_minus_bid_short,weight,is_negative
5,C -> B,C,B,0.5,1.04,0.98,-0.06,-0.030,True
0,A -> B,A,B,0.8,1.00,0.98,-0.02,-0.016,True
4,C -> A,C,A,0.2,1.04,1.01,-0.03,-0.006,True


In [43]:
def verify_market_weights(ask, bid, sim, w):
    ask = np.asarray(ask, dtype=float)
    bid = np.asarray(bid, dtype=float)
    sim = np.asarray(sim, dtype=float)
    w = np.asarray(w, dtype=float)

    N = len(ask)

    checks = {}

    checks["shape_is_N_plus_1"] = (w.shape == (N + 1, N + 1))
    checks["dummy_out_zero"] = np.allclose(w[0, :], 0.0)
    checks["dummy_in_zero"] = np.allclose(w[:, 0], 0.0)
    checks["diagonal_zero"] = np.allclose(np.diag(w), 0.0)

    formula_ok = True

    for i in range(1, N + 1):
        for j in range(1, N + 1):
            if i == j:
                expected = 0.0
            else:
                expected = sim[i - 1, j - 1] * (ask[j - 1] - bid[i - 1])

            if not np.isclose(w[i, j], expected):
                formula_ok = False

    checks["formula_correct"] = formula_ok

    # Проверяем, что хотя бы одна пара действительно направленная
    directed_pairs = []

    for i in range(1, N + 1):
        for j in range(i + 1, N + 1):
            directed_pairs.append(not np.isclose(w[i, j], w[j, i]))

    checks["has_directed_asymmetry"] = any(directed_pairs)

    return pd.DataFrame(
        [{"check": k, "passed": v} for k, v in checks.items()]
    )


market_graph_checks = verify_market_weights(ask, bid, sim, w)

market_graph_checks

,check,passed
0,shape_is_N_plus_1,True
1,dummy_out_zero,True
2,dummy_in_zero,True
3,diagonal_zero,True
4,formula_correct,True
5,has_directed_asymmetry,True


In [44]:
assert market_graph_checks["passed"].all()

print("All market graph checks passed.")

All market graph checks passed.


# 4.2.2. Проверка dummy node и ограничений цикла

В QUBO-постановке торговая пара кодируется как направленный цикл, проходящий через dummy node:

$$
0 \to \text{short} \to \ldots \to \text{long} \to 0.
$$

Узел после dummy node задаёт short-актив, а узел перед dummy node задаёт long-актив.

В этом блоке проверяется, что выбранные бинарные переменные действительно задают один корректный directed cycle через dummy node, а не произвольный набор рёбер.

In [45]:
from qubo_cycle import build_cycle_qubo
from pair_search import verify_single_dummy_cycle, extract_pair_and_weight

In [46]:
def z_from_edges(selected_edges, var_map):
    """
    Переводит список выбранных directed edges в бинарный QUBO-вектор z.
    """
    z = np.zeros(len(var_map.edges), dtype=int)

    for edge in selected_edges:
        u = var_map.e2u[edge]
        z[u] = 1

    return z


def qubo_energy(z, lin, quad):
    """
    Считает QUBO-энергию:
        E(z) = sum_i lin_i z_i + sum_{i<j} quad_ij z_i z_j
    """
    z = np.asarray(z, dtype=float).reshape(-1)

    E = float(np.dot(lin, z))

    for (u, v), q in quad.items():
        E += float(q) * z[u] * z[v]

    return E

In [ ]:
N_cycle = 5

w_zero = np.zeros((N_cycle + 1, N_cycle + 1), dtype=float)

lin_penalty, quad_penalty, var_map_cycle = build_cycle_qubo(
    w=w_zero,
    tabu=None,
    mc=0.0,
    mp=1.0,
)

len(var_map_cycle.edges)

30

In [48]:
cycle_test_cases = [
    {
        "case": "valid direct cycle",
        "edges": [(0, 1), (1, 2), (2, 0)],
        "expected_ok": True,
        "comment": "0 -> short -> long -> 0",
    },
    {
        "case": "valid bypass cycle",
        "edges": [(0, 1), (1, 3), (3, 2), (2, 0)],
        "expected_ok": True,
        "comment": "0 -> short -> intermediate -> long -> 0",
    },
    {
        "case": "cycle without dummy",
        "edges": [(1, 2), (2, 3), (3, 1)],
        "expected_ok": False,
        "comment": "cycle exists, but dummy node is absent",
    },
    {
        "case": "trivial dummy cycle",
        "edges": [(0, 1), (1, 0)],
        "expected_ok": False,
        "comment": "short and long would be the same asset",
    },
    {
        "case": "two outgoing edges from dummy",
        "edges": [(0, 1), (0, 2), (1, 0), (2, 0)],
        "expected_ok": False,
        "comment": "dummy has out-degree greater than 1",
    },
    {
        "case": "open path",
        "edges": [(0, 1), (1, 2)],
        "expected_ok": False,
        "comment": "not a closed cycle",
    },
    {
        "case": "split cycles",
        "edges": [(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3)],
        "expected_ok": False,
        "comment": "one cycle contains dummy, another cycle is disconnected",
    },
]

In [49]:
cycle_check_rows = []

for item in cycle_test_cases:
    selected_edges = item["edges"]

    ok, cycle = verify_single_dummy_cycle(
        selected_edges=selected_edges,
        N=N_cycle,
    )

    z = z_from_edges(selected_edges, var_map_cycle)
    penalty_energy = qubo_energy(z, lin_penalty, quad_penalty)

    cycle_check_rows.append({
        "case": item["case"],
        "edges": selected_edges,
        "expected_ok": item["expected_ok"],
        "actual_ok": ok,
        "cycle": cycle,
        "penalty_energy": penalty_energy,
        "comment": item["comment"],
    })

cycle_checks_df = pd.DataFrame(cycle_check_rows)

cycle_checks_df

,case,edges,expected_ok,actual_ok,cycle,penalty_energy,comment
0,valid direct cycle,"[(0, 1), (1, 2), (2, 0)]",True,True,"[0, 1, 2, 0]",0.0,0 -> short -> long -> 0
1,valid bypass cycle,"[(0, 1), (1, 3), (3, 2), (2, 0)]",True,True,"[0, 1, 3, 2, 0]",0.0,0 -> short -> intermediate -> long -> 0
2,cycle without dummy,"[(1, 2), (2, 3), (3, 1)]",False,False,None,0.0,"cycle exists, but dummy node is absent"
3,trivial dummy cycle,"[(0, 1), (1, 0)]",False,False,None,1.0,short and long would be the same asset
4,two outgoing edges from dummy,"[(0, 1), (0, 2), (1, 0), (2, 0)]",False,False,None,4.0,dummy has out-degree greater than 1
5,open path,"[(0, 1), (1, 2)]",False,False,None,2.0,not a closed cycle
6,split cycles,"[(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3)]",False,False,None,0.0,"one cycle contains dummy, another cycle is dis..."


In [50]:
assert np.all(
    cycle_checks_df["expected_ok"].values == cycle_checks_df["actual_ok"].values
)

print("Cycle verification checks passed.")

Cycle verification checks passed.


In [51]:
cycle_checks_df[
    (cycle_checks_df["actual_ok"] == False) &
    (np.isclose(cycle_checks_df["penalty_energy"], 0.0))
][
    ["case", "edges", "penalty_energy", "comment"]
]

,case,edges,penalty_energy,comment
2,cycle without dummy,"[(1, 2), (2, 3), (3, 1)]",0.0,"cycle exists, but dummy node is absent"
6,split cycles,"[(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3)]",0.0,"one cycle contains dummy, another cycle is dis..."


In [52]:
w_cycle_demo = np.zeros((N_cycle + 1, N_cycle + 1), dtype=float)

# direct path 1 -> 2
w_cycle_demo[1, 2] = -0.10

# bypass path 1 -> 3 -> 2
w_cycle_demo[1, 3] = -0.04
w_cycle_demo[3, 2] = -0.08

valid_cycle_examples = [
    [(0, 1), (1, 2), (2, 0)],
    [(0, 1), (1, 3), (3, 2), (2, 0)],
]

extract_rows = []

for selected_edges in valid_cycle_examples:
    ok, cycle = verify_single_dummy_cycle(selected_edges, N=N_cycle)

    assert ok

    short_node, long_node, path_nodes, path_weight = extract_pair_and_weight(
        cycle=cycle,
        w=w_cycle_demo,
    )

    extract_rows.append({
        "edges": selected_edges,
        "cycle": cycle,
        "short_node": short_node,
        "long_node": long_node,
        "path_nodes": path_nodes,
        "path_weight": path_weight,
    })

extract_df = pd.DataFrame(extract_rows)

extract_df

,edges,cycle,short_node,long_node,path_nodes,path_weight
0,"[(0, 1), (1, 2), (2, 0)]","[0, 1, 2, 0]",1,2,"[1, 2]",-0.10
1,"[(0, 1), (1, 3), (3, 2), (2, 0)]","[0, 1, 3, 2, 0]",1,2,"[1, 3, 2]",-0.12


In [53]:
assert extract_df.loc[0, "short_node"] == 1
assert extract_df.loc[0, "long_node"] == 2
assert np.isclose(extract_df.loc[0, "path_weight"], -0.10)

assert extract_df.loc[1, "short_node"] == 1
assert extract_df.loc[1, "long_node"] == 2
assert np.isclose(extract_df.loc[1, "path_weight"], -0.12)

print("Pair extraction checks passed.")

Pair extraction checks passed.


# 4.2.3. Проверка tabu list

После нахождения торговой пары необходимо запретить повторное нахождение той же пары в следующих запусках. Для этого используется tabu list.

Важно, что tabu list запрещает не отдельное ребро внутри пути, а именно пару endpoints:

$$
0 \to \text{short} \to \ldots \to \text{long} \to 0.
$$

Если пара имеет endpoints $(\text{short}, \text{long})$, то в матрицу tabu записывается

$$
T_{\text{long}, \text{short}} = 1.
$$

В QUBO это соответствует штрафу

$$
T_{i,j} b_{0,j} b_{i,0},
$$

где $j$ - short node, а $i$ - long node.

In [54]:
from pair_search import TabuList

In [55]:
N_tabu = 3

w_tabu_demo = np.zeros((N_tabu + 1, N_tabu + 1), dtype=float)

# Делаем все stock-stock переходы умеренно положительными
for i in range(1, N_tabu + 1):
    for j in range(1, N_tabu + 1):
        if i != j:
            w_tabu_demo[i, j] = 0.20

# Лучшее направление: short=1, long=2
w_tabu_demo[1, 2] = -0.30

# Второе лучшее направление: short=1, long=3
w_tabu_demo[1, 3] = -0.20

pd.DataFrame(
    w_tabu_demo,
    index=["dummy", "A", "B", "C"],
    columns=["dummy", "A", "B", "C"],
)

,dummy,A,B,C
dummy,0.0,0.0,0.0,0.0
A,0.0,0.0,-0.3,-0.2
B,0.0,0.2,0.0,0.2
C,0.0,0.2,0.2,0.0


In [56]:
def decode_edges_from_z(z, var_map):
    """
    Переводит бинарный QUBO-вектор z в список выбранных directed edges.
    """
    z = np.asarray(z, dtype=int).reshape(-1)

    selected_edges = []

    for u, bit in enumerate(z):
        if bit == 1:
            selected_edges.append(var_map.u2e[u])

    return selected_edges


def find_best_valid_cycle_by_qubo_bruteforce(lin, quad, var_map, N, w):
    """
    Полный перебор всех QUBO-конфигураций.
    Возвращает лучший валидный dummy-cycle по QUBO-energy.
    """
    M = len(var_map.edges)

    best = None
    rows = []

    for mask in range(1 << M):
        z = np.array(
            [(mask >> u) & 1 for u in range(M)],
            dtype=int,
        )

        selected_edges = decode_edges_from_z(z, var_map)

        ok, cycle = verify_single_dummy_cycle(
            selected_edges=selected_edges,
            N=N,
        )

        if not ok:
            continue

        E_qubo = qubo_energy(z, lin, quad)

        short_node, long_node, path_nodes, path_weight = extract_pair_and_weight(
            cycle=cycle,
            w=w,
        )

        row = {
            "E_qubo": E_qubo,
            "cycle": cycle,
            "selected_edges": selected_edges,
            "short_node": short_node,
            "long_node": long_node,
            "path_nodes": path_nodes,
            "path_weight": path_weight,
        }

        rows.append(row)

        if best is None or E_qubo < best["E_qubo"] - 1e-12:
            best = row

    all_valid_df = pd.DataFrame(rows).sort_values("E_qubo").reset_index(drop=True)

    return best, all_valid_df

In [57]:
MC_TABU = 1.0
MP_TABU = 10.0

tabu_empty = TabuList(N_tabu)

lin_no_tabu, quad_no_tabu, var_map_no_tabu = build_cycle_qubo(
    w=w_tabu_demo,
    tabu=tabu_empty.T,
    mc=MC_TABU,
    mp=MP_TABU,
)

best_no_tabu, valid_cycles_no_tabu_df = find_best_valid_cycle_by_qubo_bruteforce(
    lin=lin_no_tabu,
    quad=quad_no_tabu,
    var_map=var_map_no_tabu,
    N=N_tabu,
    w=w_tabu_demo,
)

best_no_tabu

{'E_qubo': np.float64(-0.29999999999999716),
 'cycle': [0, 1, 2, 0],
 'selected_edges': [(0, 1), (1, 2), (2, 0)],
 'short_node': 1,
 'long_node': 2,
 'path_nodes': [1, 2],
 'path_weight': -0.3}

In [58]:
valid_cycles_no_tabu_df.head(10)[
    ["E_qubo", "cycle", "short_node", "long_node", "path_nodes", "path_weight"]
]

,E_qubo,cycle,short_node,long_node,path_nodes,path_weight
0,-0.3,"[0, 1, 2, 0]",1,2,"[1, 2]",-0.3
1,-0.2,"[0, 1, 3, 0]",1,3,"[1, 3]",-0.2
2,-0.1,"[0, 3, 1, 2, 0]",3,2,"[3, 1, 2]",-0.1
3,-0.1,"[0, 1, 2, 3, 0]",1,3,"[1, 2, 3]",-0.1
4,0.0,"[0, 2, 1, 3, 0]",2,3,"[2, 1, 3]",0.0
5,0.0,"[0, 1, 3, 2, 0]",1,2,"[1, 3, 2]",0.0
6,0.2,"[0, 2, 3, 0]",2,3,"[2, 3]",0.2
7,0.2,"[0, 2, 1, 0]",2,1,"[2, 1]",0.2
8,0.2,"[0, 3, 1, 0]",3,1,"[3, 1]",0.2
9,0.2,"[0, 3, 2, 0]",3,2,"[3, 2]",0.2


In [59]:
tabu = TabuList(N_tabu)

short_found = best_no_tabu["short_node"]
long_found = best_no_tabu["long_node"]

tabu.forbid(short=short_found, long=long_found)

tabu.T

array([[0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 0]])

In [60]:
assert tabu.T[long_found, short_found] == 1
assert tabu.T[short_found, long_found] == 0

print(f"Forbidden pair: short={short_found}, long={long_found}")
print(f"Tabu entry: T[{long_found}, {short_found}] = {tabu.T[long_found, short_found]}")

Forbidden pair: short=1, long=2
Tabu entry: T[2, 1] = 1


In [61]:
old_best_edges = best_no_tabu["selected_edges"]

z_old_no_tabu = z_from_edges(old_best_edges, var_map_no_tabu)

E_old_no_tabu = qubo_energy(
    z_old_no_tabu,
    lin_no_tabu,
    quad_no_tabu,
)

lin_with_tabu, quad_with_tabu, var_map_with_tabu = build_cycle_qubo(
    w=w_tabu_demo,
    tabu=tabu.T,
    mc=MC_TABU,
    mp=MP_TABU,
)

z_old_with_tabu = z_from_edges(old_best_edges, var_map_with_tabu)

E_old_with_tabu = qubo_energy(
    z_old_with_tabu,
    lin_with_tabu,
    quad_with_tabu,
)

E_old_no_tabu, E_old_with_tabu, E_old_with_tabu - E_old_no_tabu

(np.float64(-0.29999999999999716),
 np.float64(9.700000000000003),
 np.float64(10.0))

In [62]:
assert np.isclose(E_old_with_tabu - E_old_no_tabu, MP_TABU)

print("Tabu penalty for the old pair is added correctly.")

Tabu penalty for the old pair is added correctly.


In [63]:
best_with_tabu, valid_cycles_with_tabu_df = find_best_valid_cycle_by_qubo_bruteforce(
    lin=lin_with_tabu,
    quad=quad_with_tabu,
    var_map=var_map_with_tabu,
    N=N_tabu,
    w=w_tabu_demo,
)

best_with_tabu

{'E_qubo': np.float64(-0.20000000000000284),
 'cycle': [0, 1, 3, 0],
 'selected_edges': [(0, 1), (1, 3), (3, 0)],
 'short_node': 1,
 'long_node': 3,
 'path_nodes': [1, 3],
 'path_weight': -0.2}

In [64]:
valid_cycles_with_tabu_df.head(10)[
    ["E_qubo", "cycle", "short_node", "long_node", "path_nodes", "path_weight"]
]

,E_qubo,cycle,short_node,long_node,path_nodes,path_weight
0,-0.2,"[0, 1, 3, 0]",1,3,"[1, 3]",-0.2
1,-0.1,"[0, 1, 2, 3, 0]",1,3,"[1, 2, 3]",-0.1
2,-0.1,"[0, 3, 1, 2, 0]",3,2,"[3, 1, 2]",-0.1
3,0.0,"[0, 2, 1, 3, 0]",2,3,"[2, 1, 3]",0.0
4,0.2,"[0, 2, 3, 0]",2,3,"[2, 3]",0.2
5,0.2,"[0, 2, 1, 0]",2,1,"[2, 1]",0.2
6,0.2,"[0, 3, 1, 0]",3,1,"[3, 1]",0.2
7,0.2,"[0, 3, 2, 0]",3,2,"[3, 2]",0.2
8,0.4,"[0, 3, 2, 1, 0]",3,1,"[3, 2, 1]",0.4
9,0.4,"[0, 2, 3, 1, 0]",2,1,"[2, 3, 1]",0.4


In [65]:
assert best_with_tabu["short_node"] == 1
assert best_with_tabu["long_node"] == 3
assert np.isclose(best_with_tabu["path_weight"], -0.20)

assert not (
    best_with_tabu["short_node"] == short_found
    and best_with_tabu["long_node"] == long_found
)

print("Tabu successfully prevents rediscovery of the same pair.")

Tabu successfully prevents rediscovery of the same pair.


In [66]:
non_endpoint_use_edges = [(0, 3), (3, 1), (1, 2), (2, 0)]

ok, cycle = verify_single_dummy_cycle(non_endpoint_use_edges, N=N_tabu)
assert ok

short_node, long_node, path_nodes, path_weight = extract_pair_and_weight(
    cycle=cycle,
    w=w_tabu_demo,
)

short_node, long_node, path_nodes, path_weight

(3, 2, [3, 1, 2], -0.09999999999999998)

In [67]:
z_non_endpoint_no_tabu = z_from_edges(non_endpoint_use_edges, var_map_no_tabu)
z_non_endpoint_with_tabu = z_from_edges(non_endpoint_use_edges, var_map_with_tabu)

E_non_endpoint_no_tabu = qubo_energy(
    z_non_endpoint_no_tabu,
    lin_no_tabu,
    quad_no_tabu,
)

E_non_endpoint_with_tabu = qubo_energy(
    z_non_endpoint_with_tabu,
    lin_with_tabu,
    quad_with_tabu,
)

E_non_endpoint_no_tabu, E_non_endpoint_with_tabu, E_non_endpoint_with_tabu - E_non_endpoint_no_tabu

(np.float64(-0.09999999999999432),
 np.float64(-0.09999999999999432),
 np.float64(0.0))

In [68]:
assert np.isclose(E_non_endpoint_with_tabu - E_non_endpoint_no_tabu, 0.0)

print("Tabu affects only endpoints, not internal edges.")

Tabu affects only endpoints, not internal edges.


In [69]:
tabu_summary = pd.DataFrame([
    {
        "check": "best pair before tabu",
        "passed": (
            best_no_tabu["short_node"] == 1
            and best_no_tabu["long_node"] == 2
            and np.isclose(best_no_tabu["path_weight"], -0.30)
        ),
        "details": str(best_no_tabu["cycle"]),
    },
    {
        "check": "tabu matrix entry",
        "passed": bool(tabu.T[2, 1] == 1 and tabu.T[1, 2] == 0),
        "details": "T[long, short] = T[2,1] = 1",
    },
    {
        "check": "old pair receives penalty",
        "passed": np.isclose(E_old_with_tabu - E_old_no_tabu, MP_TABU),
        "details": f"penalty = {E_old_with_tabu - E_old_no_tabu}",
    },
    {
        "check": "best pair after tabu changes",
        "passed": (
            best_with_tabu["short_node"] == 1
            and best_with_tabu["long_node"] == 3
            and np.isclose(best_with_tabu["path_weight"], -0.20)
        ),
        "details": str(best_with_tabu["cycle"]),
    },
    {
        "check": "internal edge is not forbidden",
        "passed": np.isclose(E_non_endpoint_with_tabu - E_non_endpoint_no_tabu, 0.0),
        "details": "cycle 0 -> 3 -> 1 -> 2 -> 0 uses edge 1->2 internally",
    },
])

tabu_summary

,check,passed,details
0,best pair before tabu,True,"[0, 1, 2, 0]"
1,tabu matrix entry,True,"T[long, short] = T[2,1] = 1"
2,old pair receives penalty,True,penalty = 10.0
3,best pair after tabu changes,True,"[0, 1, 3, 0]"
4,internal edge is not forbidden,True,cycle 0 -> 3 -> 1 -> 2 -> 0 uses edge 1->2 int...


In [70]:
assert tabu_summary["passed"].all()

print("All tabu list checks passed.")

All tabu list checks passed.


# 4.2.4. Проверка эквивалентности QUBO и Ising

После построения QUBO-задачи она преобразуется в Ising-форму, поскольку SB-алгоритмы работают с Ising-энергией.

QUBO-задача имеет вид

$$
E_{\mathrm{QUBO}}(z) = \sum_i q_i z_i + \sum_{i<j} q_{ij}z_i z_j, \qquad z_i \in \{0,1\}.
$$

Ising-переменные связаны с бинарными переменными соотношением

$$
z_i = \frac{s_i+1}{2}, \qquad s_i \in \{-1,+1\}. 
$$

После преобразования должна выполняться эквивалентность

$$
E_{\mathrm{QUBO}}(z) = E_{\mathrm{Ising}}(s) + C,
$$

где $C$ - константа, не влияющая на положение минимума.

В этом блоке проверяется, что QUBO- и Ising-представления дают одинаковые значения энергии для всех бинарных конфигураций малой задачи.

In [71]:
from ising_mapping import qubo_to_ising

In [72]:
def mapped_ising_energy(s, J, h, const=0.0):
    """
    Энергия Ising-представления после qubo_to_ising:

        E(s) = - sum_{i<j} J_ij s_i s_j - sum_i h_i s_i + const

    Здесь J может быть dense или sparse.
    """
    s = np.asarray(s, dtype=float).reshape(-1)

    if hasattr(J, "toarray"):
        J_dense = J.toarray()
    else:
        J_dense = np.asarray(J, dtype=float)

    h = np.asarray(h, dtype=float).reshape(-1)

    pair_term = 0.0
    N = len(s)

    for i in range(N):
        for j in range(i + 1, N):
            pair_term += J_dense[i, j] * s[i] * s[j]

    field_term = np.dot(h, s)

    return float(-pair_term - field_term + const)

In [73]:
N_equiv = 3

w_equiv = np.zeros((N_equiv + 1, N_equiv + 1), dtype=float)

# Несимметричные веса между stock nodes
w_equiv[1, 2] = -0.30
w_equiv[2, 1] =  0.15

w_equiv[1, 3] = -0.20
w_equiv[3, 1] =  0.10

w_equiv[2, 3] =  0.05
w_equiv[3, 2] = -0.12

# Добавим tabu на пару short=1, long=2
tabu_equiv = TabuList(N_equiv)
tabu_equiv.forbid(short=1, long=2)

MC_EQUIV = 1.0
MP_EQUIV = 10.0

lin_equiv, quad_equiv, var_map_equiv = build_cycle_qubo(
    w=w_equiv,
    tabu=tabu_equiv.T,
    mc=MC_EQUIV,
    mp=MP_EQUIV,
)

J_equiv, h_equiv, const_equiv = qubo_to_ising(
    lin=lin_equiv,
    quad=quad_equiv,
)

M_equiv = len(var_map_equiv.edges)

M_equiv, 2 ** M_equiv, const_equiv

(12, 4096, 137.34000000000003)

In [74]:
equivalence_rows = []

max_abs_error = 0.0
worst_mask = None

for mask in range(1 << M_equiv):
    z = np.array(
        [(mask >> u) & 1 for u in range(M_equiv)],
        dtype=int,
    )

    # z = (s + 1) / 2  =>  s = 2z - 1
    s = 2 * z - 1

    E_qubo = qubo_energy(
        z=z,
        lin=lin_equiv,
        quad=quad_equiv,
    )

    E_ising_plus_const = mapped_ising_energy(
        s=s,
        J=J_equiv,
        h=h_equiv,
        const=const_equiv,
    )

    abs_error = abs(E_qubo - E_ising_plus_const)

    if abs_error > max_abs_error:
        max_abs_error = abs_error
        worst_mask = mask

    equivalence_rows.append({
        "mask": mask,
        "E_qubo": E_qubo,
        "E_ising_plus_const": E_ising_plus_const,
        "abs_error": abs_error,
    })

equivalence_df = pd.DataFrame(equivalence_rows)

equivalence_df.head()

,mask,E_qubo,E_ising_plus_const,abs_error
0,0,0.0,5.684342e-14,5.684342e-14
1,1,20.0,2.000000e+01,4.263256e-14
2,2,20.0,2.000000e+01,5.684342e-14
3,3,70.0,7.000000e+01,4.263256e-14
4,4,20.0,2.000000e+01,5.684342e-14


In [75]:
max_abs_error, worst_mask

(np.float64(1.1368683772161603e-13), 1463)

In [76]:
assert max_abs_error < 1e-8

print("QUBO and Ising energies are equivalent up to numerical precision.")

QUBO and Ising energies are equivalent up to numerical precision.


In [77]:
equivalence_df["abs_error"].describe()

count    4.096000e+03
mean     3.412982e-14
std      2.114361e-14
min      0.000000e+00
25%      2.842171e-14
50%      2.842171e-14
75%      5.684342e-14
max      1.136868e-13
Name: abs_error, dtype: float64

In [78]:
E_qubo_min = equivalence_df["E_qubo"].min()
E_ising_min = equivalence_df["E_ising_plus_const"].min()

qubo_min_masks = set(
    equivalence_df.loc[
        np.isclose(equivalence_df["E_qubo"], E_qubo_min),
        "mask",
    ].tolist()
)

ising_min_masks = set(
    equivalence_df.loc[
        np.isclose(equivalence_df["E_ising_plus_const"], E_ising_min),
        "mask",
    ].tolist()
)

E_qubo_min, E_ising_min, len(qubo_min_masks), len(ising_min_masks)

(np.float64(-0.25), np.float64(-0.24999999999997158), 1, 1)

In [79]:
assert np.isclose(E_qubo_min, E_ising_min)
assert qubo_min_masks == ising_min_masks

print("QUBO and Ising minima coincide.")

QUBO and Ising minima coincide.


In [80]:
best_mask = min(qubo_min_masks)

z_best = np.array(
    [(best_mask >> u) & 1 for u in range(M_equiv)],
    dtype=int,
)

s_best = 2 * z_best - 1

selected_edges_best = decode_edges_from_z(
    z_best,
    var_map_equiv,
)

ok_best, cycle_best = verify_single_dummy_cycle(
    selected_edges=selected_edges_best,
    N=N_equiv,
)

E_best_qubo = qubo_energy(
    z=z_best,
    lin=lin_equiv,
    quad=quad_equiv,
)

E_best_ising = mapped_ising_energy(
    s=s_best,
    J=J_equiv,
    h=h_equiv,
    const=const_equiv,
)

best_decoding = {
    "best_mask": best_mask,
    "E_best_qubo": E_best_qubo,
    "E_best_ising_plus_const": E_best_ising,
    "selected_edges": selected_edges_best,
    "is_valid_dummy_cycle": ok_best,
    "cycle": cycle_best,
}

best_decoding

{'best_mask': 785,
 'E_best_qubo': np.float64(-0.25),
 'E_best_ising_plus_const': -0.24999999999997158,
 'selected_edges': [(0, 1), (1, 2), (2, 3), (3, 0)],
 'is_valid_dummy_cycle': True,
 'cycle': [0, 1, 2, 3, 0]}

In [81]:
top_qubo = equivalence_df.sort_values("E_qubo").head(10)[
    ["mask", "E_qubo", "E_ising_plus_const", "abs_error"]
]

top_ising = equivalence_df.sort_values("E_ising_plus_const").head(10)[
    ["mask", "E_qubo", "E_ising_plus_const", "abs_error"]
]

top_qubo

,mask,E_qubo,E_ising_plus_const,abs_error
785,785,-0.25,-2.500000e-01,2.842171e-14
545,545,-0.20,-2.000000e-01,4.263256e-14
1108,1108,-0.20,-2.000000e-01,2.842171e-14
2208,2208,-0.17,-1.700000e-01,3.552714e-14
1296,1296,-0.15,-1.500000e-01,2.131628e-14
2116,2116,-0.12,-1.200000e-01,4.973799e-14
674,674,-0.05,-5.000000e-02,5.684342e-14
0,0,0.00,5.684342e-14,5.684342e-14
2188,2188,0.03,3.000000e-02,2.842171e-14
770,770,0.05,5.000000e-02,1.421085e-14


In [82]:
top_ising

,mask,E_qubo,E_ising_plus_const,abs_error
785,785,-0.25,-2.500000e-01,2.842171e-14
545,545,-0.20,-2.000000e-01,4.263256e-14
1108,1108,-0.20,-2.000000e-01,2.842171e-14
2208,2208,-0.17,-1.700000e-01,3.552714e-14
1296,1296,-0.15,-1.500000e-01,2.131628e-14
2116,2116,-0.12,-1.200000e-01,4.973799e-14
674,674,-0.05,-5.000000e-02,5.684342e-14
0,0,0.00,5.684342e-14,5.684342e-14
2188,2188,0.03,3.000000e-02,2.842171e-14
770,770,0.05,5.000000e-02,1.421085e-14


In [83]:
qubo_ising_summary = pd.DataFrame([
    {
        "check": "number of configurations",
        "passed": (len(equivalence_df) == 2 ** M_equiv),
        "details": f"{len(equivalence_df)} configurations checked",
    },
    {
        "check": "energy equivalence",
        "passed": (max_abs_error < 1e-8),
        "details": f"max_abs_error = {max_abs_error:.3e}",
    },
    {
        "check": "minimum energy equality",
        "passed": np.isclose(E_qubo_min, E_ising_min),
        "details": f"E_min = {E_qubo_min:.6f}",
    },
    {
        "check": "argmin set equality",
        "passed": (qubo_min_masks == ising_min_masks),
        "details": f"n_minimizers = {len(qubo_min_masks)}",
    },
    {
        "check": "top-10 order equality",
        "passed": (top_qubo["mask"].tolist() == top_ising["mask"].tolist()),
        "details": "top-10 masks coincide",
    },
])

qubo_ising_summary

,check,passed,details
0,number of configurations,True,4096 configurations checked
1,energy equivalence,True,max_abs_error = 1.137e-13
2,minimum energy equality,True,E_min = -0.250000
3,argmin set equality,True,n_minimizers = 1
4,top-10 order equality,True,top-10 masks coincide


In [84]:
assert qubo_ising_summary["passed"].all()

print("All QUBO-Ising equivalence checks passed.")

All QUBO-Ising equivalence checks passed.


In [85]:
market_graph_checks.to_csv(
    "checks_04_02_01_market_graph.csv",
    index=False,
)

cycle_checks_df.to_csv(
    "checks_04_02_02_dummy_cycle.csv",
    index=False,
)

tabu_summary.to_csv(
    "checks_04_02_03_tabu.csv",
    index=False,
)

qubo_ising_summary.to_csv(
    "checks_04_02_04_qubo_ising_equivalence.csv",
    index=False,
)

print("Saved all 4.2 verification checks.")

Saved all 4.2 verification checks.


In [86]:
same_endpoint_bypass_edges = [(0, 1), (1, 3), (3, 2), (2, 0)]

z_same_endpoint_no_tabu = z_from_edges(same_endpoint_bypass_edges, var_map_no_tabu)
z_same_endpoint_with_tabu = z_from_edges(same_endpoint_bypass_edges, var_map_with_tabu)

E_same_endpoint_no_tabu = qubo_energy(
    z_same_endpoint_no_tabu,
    lin_no_tabu,
    quad_no_tabu,
)

E_same_endpoint_with_tabu = qubo_energy(
    z_same_endpoint_with_tabu,
    lin_with_tabu,
    quad_with_tabu,
)

assert np.isclose(
    E_same_endpoint_with_tabu - E_same_endpoint_no_tabu,
    MP_TABU,
)